<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/comunicacio/ca/identitat/descarrega-arxius-grafics/fitxers-marca-principal/upc-positiu-p3005.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 10px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Model Risk & Data Science Training</b></p>
      <p style="margin: 0;">Python profesional para analítica avanzada</p>
  </div>

</div>

# **Testing, calidad de código, pytest, Ruff, Black, mypy y packaging**

## Del código que “parece funcionar” al código verificable

En este laboratorio trabajaremos una idea central:

> **el código profesional no solo se ejecuta: también debe poder demostrarse que se comporta como esperamos.**

La secuencia será:

> **introducir bug → escribir test → observar fallo → diagnosticar → corregir → volver a ejecutar → automatizar calidad**

Integraremos:

- tests unitarios;
- pytest;
- asserts;
- parametrización;
- fixtures;
- tests de excepciones;
- tests de DataFrames;
- regresiones;
- linting;
- formateo;
- type checking;
- packaging;
- instalación editable;
- estructura `src/`;
- proyecto final verificable.

## Objetivos de aprendizaje

Al finalizar deberías poder:

1. Explicar qué aporta un test.
2. Diferenciar test unitario, integración y regresión.
3. Escribir funciones testeables.
4. Utilizar `assert`.
5. Utilizar pytest.
6. Interpretar un test que falla.
7. Parametrizar tests.
8. Utilizar fixtures.
9. Testear excepciones.
10. Testear DataFrames.
11. Detectar bugs de regresión.
12. Comprender cobertura de tests.
13. Utilizar Ruff como linter.
14. Utilizar Black como formatter.
15. Utilizar mypy para chequeo estático.
16. Entender el papel de `pyproject.toml`.
17. Crear un paquete instalable.
18. Ejecutar calidad sobre un proyecto completo.

## 1. Preparación

In [ ]:
from pathlib import Path
import subprocess
import sys
import textwrap
import shutil

import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)

print("Python:", sys.version)

## 2. Crear un proyecto de testing

In [ ]:
raiz = Path("proyecto_testing_demo")

if raiz.exists():
    shutil.rmtree(raiz)

for carpeta in [
    raiz / "src" / "proyecto_testing",
    raiz / "tests"
]:
    carpeta.mkdir(parents=True, exist_ok=True)

(raiz / "src" / "proyecto_testing" / "__init__.py").write_text(
    "",
    encoding="utf-8"
)

(raiz / "tests" / "__init__.py").write_text(
    "",
    encoding="utf-8"
)

print("Proyecto creado en:", raiz.resolve())

## 2.1 Inspeccionar estructura

In [ ]:
for ruta in sorted(raiz.rglob("*")):
    nivel = len(ruta.relative_to(raiz).parts) - 1
    sangria = "    " * nivel
    simbolo = "📁" if ruta.is_dir() else "📄"
    print(f"{sangria}{simbolo} {ruta.name}")

## 3. Primera función con un bug deliberado

In [ ]:
codigo_bug = '''
def calcular_descuento(importe: float, porcentaje: float) -> float:
    # BUG deliberado: suma en lugar de restar el descuento.
    return importe * (1 + porcentaje)
'''

ruta_pricing = raiz / "src" / "proyecto_testing" / "pricing.py"

ruta_pricing.write_text(
    textwrap.dedent(codigo_bug),
    encoding="utf-8"
)

print(ruta_pricing.read_text(encoding="utf-8"))

### Observación

La función es sintácticamente correcta y produce un número.

El problema es **semántico**: calcula mal.

## 4. Test manual

In [ ]:
def calcular_descuento_bug(importe, porcentaje):
    return importe * (1 + porcentaje)

esperado = 80.0
obtenido = calcular_descuento_bug(100, 0.20)

print("Esperado:", esperado)
print("Obtenido:", obtenido)

try:
    assert np.isclose(obtenido, esperado)
except AssertionError:
    print("TEST FALLIDO: el resultado no coincide.")

El `assert` convierte una expectativa en una condición verificable.

## 5. Crear primer test con pytest

In [ ]:
test_pricing = '''
from proyecto_testing.pricing import calcular_descuento

def test_calcular_descuento():
    resultado = calcular_descuento(100, 0.20)
    assert resultado == 80
'''

ruta_test_pricing = raiz / "tests" / "test_pricing.py"

ruta_test_pricing.write_text(
    textwrap.dedent(test_pricing),
    encoding="utf-8"
)

print(ruta_test_pricing.read_text(encoding="utf-8"))

## 5.1 Detectar pytest

In [ ]:
pytest_disponible = shutil.which("pytest") is not None

print("pytest disponible:", pytest_disponible)

## 5.2 Ejecutar y observar el fallo

In [ ]:
if pytest_disponible:
    resultado = subprocess.run(
        ["pytest", "-q"],
        cwd=raiz,
        capture_output=True,
        text=True
    )

    print(resultado.stdout)
    print(resultado.stderr)
    print("Return code:", resultado.returncode)
else:
    print("Instala pytest para ejecutar esta sección.")

### Interpretación

Aquí queremos que el test falle.

Un test rojo es información: demuestra que el comportamiento real no coincide con el esperado.

## 6. Corregir la función

In [ ]:
codigo_corregido = '''
def calcular_descuento(importe: float, porcentaje: float) -> float:
    return importe * (1 - porcentaje)
'''

ruta_pricing.write_text(
    textwrap.dedent(codigo_corregido),
    encoding="utf-8"
)

print(ruta_pricing.read_text(encoding="utf-8"))

## 6.1 Volver a ejecutar

In [ ]:
if pytest_disponible:
    resultado = subprocess.run(
        ["pytest", "-q"],
        cwd=raiz,
        capture_output=True,
        text=True
    )

    print(resultado.stdout)
    print("Return code:", resultado.returncode)

### Patrón metodológico

1. bug;
2. test;
3. fallo;
4. corrección;
5. test verde.

## 7. Añadir validación a la función

In [ ]:
codigo_validado = '''
def calcular_descuento(importe: float, porcentaje: float) -> float:
    if importe < 0:
        raise ValueError("importe no puede ser negativo")

    if not 0 <= porcentaje <= 1:
        raise ValueError("porcentaje debe estar entre 0 y 1")

    return importe * (1 - porcentaje)
'''

ruta_pricing.write_text(
    textwrap.dedent(codigo_validado),
    encoding="utf-8"
)

## 7.1 Tests de excepciones

In [ ]:
test_excepciones = '''
import pytest

from proyecto_testing.pricing import calcular_descuento

def test_importe_negativo():
    with pytest.raises(ValueError, match="importe"):
        calcular_descuento(-100, 0.2)

def test_porcentaje_fuera_rango():
    with pytest.raises(ValueError, match="porcentaje"):
        calcular_descuento(100, 1.5)
'''

(raiz / "tests" / "test_pricing_errors.py").write_text(
    textwrap.dedent(test_excepciones),
    encoding="utf-8"
)

## 7.2 Ejecutar

In [ ]:
if pytest_disponible:
    resultado = subprocess.run(
        ["pytest", "-q"],
        cwd=raiz,
        capture_output=True,
        text=True
    )
    print(resultado.stdout)

## 8. Parametrización

In [ ]:
test_param = '''
import pytest

from proyecto_testing.pricing import calcular_descuento

@pytest.mark.parametrize(
    "importe, porcentaje, esperado",
    [
        (100, 0.0, 100),
        (100, 0.2, 80),
        (200, 0.5, 100),
        (50, 1.0, 0),
    ]
)
def test_calcular_descuento_parametrizado(
    importe,
    porcentaje,
    esperado
):
    assert calcular_descuento(importe, porcentaje) == esperado
'''

(raiz / "tests" / "test_pricing_param.py").write_text(
    textwrap.dedent(test_param),
    encoding="utf-8"
)

Parametrizar evita duplicar muchos tests casi idénticos.

## 9. Fixtures

In [ ]:
conftest = '''
import pandas as pd
import pytest

@pytest.fixture
def ventas_demo():
    return pd.DataFrame({
        "producto": ["A", "A", "B"],
        "unidades": [2, 3, 1],
        "precio": [10.0, 20.0, 50.0]
    })
'''

(raiz / "tests" / "conftest.py").write_text(
    textwrap.dedent(conftest),
    encoding="utf-8"
)

## 10. Crear módulo de ventas

In [ ]:
codigo_ventas = '''
import pandas as pd

def calcular_importe(datos: pd.DataFrame) -> pd.DataFrame:
    salida = datos.copy()
    salida["importe"] = salida["unidades"] * salida["precio"]
    return salida

def resumir_producto(datos: pd.DataFrame) -> pd.DataFrame:
    return (
        datos.groupby("producto", as_index=False)
        .agg(
            facturacion=("importe", "sum"),
            operaciones=("importe", "size")
        )
    )
'''

(raiz / "src" / "proyecto_testing" / "ventas.py").write_text(
    textwrap.dedent(codigo_ventas),
    encoding="utf-8"
)

## 10.1 Test con fixture

In [ ]:
test_fixture = '''
from proyecto_testing.ventas import calcular_importe, resumir_producto

def test_calcular_importe(ventas_demo):
    resultado = calcular_importe(ventas_demo)

    assert "importe" in resultado.columns
    assert resultado["importe"].tolist() == [20.0, 60.0, 50.0]

def test_resumen_producto(ventas_demo):
    preparados = calcular_importe(ventas_demo)
    resumen = resumir_producto(preparados)

    assert len(resumen) == 2
'''

(raiz / "tests" / "test_ventas.py").write_text(
    textwrap.dedent(test_fixture),
    encoding="utf-8"
)

## 10.2 Test de DataFrame completo

In [ ]:
test_df = '''
import pandas as pd
from pandas.testing import assert_frame_equal

from proyecto_testing.ventas import calcular_importe, resumir_producto

def test_dataframe_completo(ventas_demo):
    resultado = (
        resumir_producto(
            calcular_importe(ventas_demo)
        )
        .sort_values("producto")
        .reset_index(drop=True)
    )

    esperado = pd.DataFrame({
        "producto": ["A", "B"],
        "facturacion": [80.0, 50.0],
        "operaciones": [2, 1]
    })

    assert_frame_equal(resultado, esperado)
'''

(raiz / "tests" / "test_dataframe.py").write_text(
    textwrap.dedent(test_df),
    encoding="utf-8"
)

Para estructuras complejas conviene utilizar utilidades específicas como `assert_frame_equal`.

## 11. Crear pipeline

In [ ]:
codigo_pipeline = '''
import pandas as pd

from .ventas import calcular_importe, resumir_producto

def ejecutar_pipeline(datos: pd.DataFrame) -> pd.DataFrame:
    preparados = calcular_importe(datos)
    return resumir_producto(preparados)
'''

(raiz / "src" / "proyecto_testing" / "pipeline.py").write_text(
    textwrap.dedent(codigo_pipeline),
    encoding="utf-8"
)

## 11.1 Test de integración

In [ ]:
test_integracion = '''
import pandas as pd
from pandas.testing import assert_frame_equal

from proyecto_testing.pipeline import ejecutar_pipeline

def test_pipeline_completo(ventas_demo):
    resultado = (
        ejecutar_pipeline(ventas_demo)
        .sort_values("producto")
        .reset_index(drop=True)
    )

    esperado = pd.DataFrame({
        "producto": ["A", "B"],
        "facturacion": [80.0, 50.0],
        "operaciones": [2, 1]
    })

    assert_frame_equal(resultado, esperado)
'''

(raiz / "tests" / "test_pipeline.py").write_text(
    textwrap.dedent(test_integracion),
    encoding="utf-8"
)

## 12. Test de regresión

In [ ]:
test_regresion = '''
from proyecto_testing.pricing import calcular_descuento

def test_regresion_descuento_20_por_ciento():
    assert calcular_descuento(100, 0.20) == 80
'''

(raiz / "tests" / "test_regression.py").write_text(
    textwrap.dedent(test_regresion),
    encoding="utf-8"
)

## 12.1 Provocar una regresión

In [ ]:
codigo_regresion = '''
def calcular_descuento(importe: float, porcentaje: float) -> float:
    if importe < 0:
        raise ValueError("importe no puede ser negativo")

    if not 0 <= porcentaje <= 1:
        raise ValueError("porcentaje debe estar entre 0 y 1")

    # REGRESIÓN deliberada
    return importe - porcentaje
'''

ruta_pricing.write_text(
    textwrap.dedent(codigo_regresion),
    encoding="utf-8"
)

## 12.2 Ejecutar y observar

In [ ]:
if pytest_disponible:
    resultado = subprocess.run(
        ["pytest", "-q"],
        cwd=raiz,
        capture_output=True,
        text=True
    )

    print(resultado.stdout)
    print("Return code:", resultado.returncode)

## 12.3 Corregir regresión

In [ ]:
ruta_pricing.write_text(
    textwrap.dedent(codigo_validado),
    encoding="utf-8"
)

print("Regresión corregida.")

## 12.4 Suite verde

In [ ]:
if pytest_disponible:
    resultado = subprocess.run(
        ["pytest", "-q"],
        cwd=raiz,
        capture_output=True,
        text=True
    )
    print(resultado.stdout)

## 13. Cobertura de tests

La cobertura indica qué partes del código han sido ejecutadas por los tests.

No garantiza calidad.

Un proyecto puede tener 100 % de cobertura y aun así tener tests pobres.

In [ ]:
print(
    'Comando recomendado: python -m pytest --cov=proyecto_testing --cov-report=term-missing'
)

## 14. Ruff: linting

In [ ]:
codigo_malo_calidad = '''
import os
import sys

def funcion_mala( x,y ):
    z=x+y
    return z
'''

ruta_calidad = raiz / "src" / "proyecto_testing" / "calidad.py"

ruta_calidad.write_text(
    textwrap.dedent(codigo_malo_calidad),
    encoding="utf-8"
)

print(ruta_calidad.read_text(encoding="utf-8"))

Problemas visibles:

- imports no usados;
- formato inconsistente;
- nombre poco descriptivo.

## 14.1 Detectar Ruff

In [ ]:
ruff_disponible = shutil.which("ruff") is not None

print("Ruff disponible:", ruff_disponible)

## 14.2 Ejecutar Ruff

In [ ]:
if ruff_disponible:
    resultado = subprocess.run(
        ["ruff", "check", "."],
        cwd=raiz,
        capture_output=True,
        text=True
    )

    print(resultado.stdout)
    print(resultado.stderr)
else:
    print("Instala Ruff para ejecutar esta sección.")

## 14.3 Ruff autofix

In [ ]:
if ruff_disponible:
    resultado = subprocess.run(
        ["ruff", "check", ".", "--fix"],
        cwd=raiz,
        capture_output=True,
        text=True
    )

    print(resultado.stdout)
    print(ruta_calidad.read_text(encoding="utf-8"))

## 15. Black: formateo

In [ ]:
black_disponible = shutil.which("black") is not None

print("Black disponible:", black_disponible)

## 15.1 Ejecutar Black

In [ ]:
if black_disponible:
    resultado = subprocess.run(
        ["black", "."],
        cwd=raiz,
        capture_output=True,
        text=True
    )

    print(resultado.stdout)
    print(resultado.stderr)
    print(ruta_calidad.read_text(encoding="utf-8"))
else:
    print("Instala Black para ejecutar esta sección.")

### Ruff vs Black

- Ruff: linting y algunas correcciones.
- Black: formateo automático consistente.

## 16. mypy: chequeo estático

In [ ]:
codigo_tipos = '''
def multiplicar(
    a: int,
    b: int
) -> int:
    return str(a * b)
'''

ruta_tipos = raiz / "src" / "proyecto_testing" / "tipos.py"

ruta_tipos.write_text(
    textwrap.dedent(codigo_tipos),
    encoding="utf-8"
)

## 16.1 Detectar mypy

In [ ]:
mypy_disponible = shutil.which("mypy") is not None

print("mypy disponible:", mypy_disponible)

## 16.2 Ejecutar mypy y observar

In [ ]:
if mypy_disponible:
    resultado = subprocess.run(
        ["mypy", "src"],
        cwd=raiz,
        capture_output=True,
        text=True
    )

    print(resultado.stdout)
    print(resultado.stderr)
else:
    print("Instala mypy para ejecutar esta sección.")

Python permitiría ejecutar la función.

Mypy detecta la contradicción entre `-> int` y `return str(...)`.

## 16.3 Corregir

In [ ]:
codigo_tipos_correcto = '''
def multiplicar(
    a: int,
    b: int
) -> int:
    return a * b
'''

ruta_tipos.write_text(
    textwrap.dedent(codigo_tipos_correcto),
    encoding="utf-8"
)

## 16.4 Reejecutar

In [ ]:
if mypy_disponible:
    resultado = subprocess.run(
        ["mypy", "src"],
        cwd=raiz,
        capture_output=True,
        text=True
    )

    print(resultado.stdout)

## 17. Test de efectos secundarios

In [ ]:
codigo_mutante = '''
import pandas as pd

def añadir_importe_mal(
    datos: pd.DataFrame
) -> pd.DataFrame:
    datos["importe"] = datos["unidades"] * datos["precio"]
    return datos
'''

ruta_mutaciones = raiz / "src" / "proyecto_testing" / "mutaciones.py"

ruta_mutaciones.write_text(
    textwrap.dedent(codigo_mutante),
    encoding="utf-8"
)

## 17.1 Test que detecta la mutación

In [ ]:
test_mutacion = '''
import pandas as pd

from proyecto_testing.mutaciones import añadir_importe_mal

def test_no_modificar_original():
    original = pd.DataFrame({
        "unidades": [2, 3],
        "precio": [10.0, 20.0]
    })

    copia = original.copy(deep=True)

    _ = añadir_importe_mal(original)

    assert original.equals(copia)
'''

(raiz / "tests" / "test_mutaciones.py").write_text(
    textwrap.dedent(test_mutacion),
    encoding="utf-8"
)

## 17.2 Ejecutar y observar fallo

In [ ]:
if pytest_disponible:
    resultado = subprocess.run(
        ["pytest", "-q"],
        cwd=raiz,
        capture_output=True,
        text=True
    )

    print(resultado.stdout)

## 17.3 Corregir efecto secundario

In [ ]:
codigo_mutante_corregido = '''
import pandas as pd

def añadir_importe_mal(
    datos: pd.DataFrame
) -> pd.DataFrame:
    salida = datos.copy()
    salida["importe"] = salida["unidades"] * salida["precio"]
    return salida
'''

ruta_mutaciones.write_text(
    textwrap.dedent(codigo_mutante_corregido),
    encoding="utf-8"
)

## 17.4 Suite verde de nuevo

In [ ]:
if pytest_disponible:
    resultado = subprocess.run(
        ["pytest", "-q"],
        cwd=raiz,
        capture_output=True,
        text=True
    )

    print(resultado.stdout)

## 18. Crear `pyproject.toml`

In [ ]:
pyproject = '''
[project]
name = "proyecto-testing"
version = "0.1.0"
description = "Proyecto de ejemplo con testing y calidad"
requires-python = ">=3.11"

dependencies = [
    "pandas>=2.2",
    "numpy>=1.26"
]

[project.optional-dependencies]
dev = [
    "pytest>=8.0",
    "pytest-cov>=5.0",
    "ruff>=0.6",
    "black>=24.0",
    "mypy>=1.10"
]

[build-system]
requires = ["setuptools>=69"]
build-backend = "setuptools.build_meta"

[tool.setuptools.packages.find]
where = ["src"]

[tool.pytest.ini_options]
testpaths = ["tests"]

[tool.ruff]
line-length = 100

[tool.black]
line-length = 100

[tool.mypy]
python_version = "3.11"
warn_return_any = true
disallow_untyped_defs = true
'''

(raiz / "pyproject.toml").write_text(
    textwrap.dedent(pyproject).strip() + "\n",
    encoding="utf-8"
)

print(
    (raiz / "pyproject.toml").read_text(
        encoding="utf-8"
    )
)

Con la estructura `src/` y `pyproject.toml`, el proyecto puede instalarse con:

```bash
python -m pip install -e .
```

Para desarrollo:

```bash
python -m pip install -e ".[dev]"
```

## 19. Crear script de calidad

In [ ]:
script_calidad = '''
from pathlib import Path
import shutil
import subprocess

comandos = [
    ["pytest", "-q"],
    ["ruff", "check", "."],
    ["black", "--check", "."],
    ["mypy", "src"],
]

for comando in comandos:
    ejecutable = comando[0]

    if shutil.which(ejecutable) is None:
        print(f"OMITIDO: {ejecutable} no está instalado.")
        continue

    print("$", " ".join(comando))

    resultado = subprocess.run(
        comando,
        capture_output=True,
        text=True
    )

    print(resultado.stdout)

    if resultado.returncode != 0:
        print(resultado.stderr)
        raise SystemExit(resultado.returncode)

print("Comprobaciones disponibles superadas.")
'''

(raiz / "check_quality.py").write_text(
    textwrap.dedent(script_calidad),
    encoding="utf-8"
)

print(
    (raiz / "check_quality.py").read_text(
        encoding="utf-8"
    )
)

El script convierte varias comprobaciones manuales en un proceso automatizable.

Más adelante podría ejecutarse en CI.

## 19.1 Ejecutar el script

In [ ]:
resultado = subprocess.run(
    [sys.executable, "check_quality.py"],
    cwd=raiz,
    capture_output=True,
    text=True
)

print(resultado.stdout)
print(resultado.stderr)
print("Return code:", resultado.returncode)

## 20. Inspeccionar estructura final

In [ ]:
for ruta in sorted(raiz.rglob("*")):
    nivel = len(ruta.relative_to(raiz).parts) - 1
    sangria = "    " * nivel
    simbolo = "📁" if ruta.is_dir() else "📄"
    print(f"{sangria}{simbolo} {ruta.name}")

## 21. Tabla de herramientas

In [ ]:
herramientas = pd.DataFrame({
    "herramienta": [
        "pytest",
        "pytest-cov",
        "Ruff",
        "Black",
        "mypy"
    ],
    "objetivo": [
        "Tests",
        "Cobertura",
        "Linting",
        "Formato",
        "Tipos"
    ],
    "corrige_codigo": [
        False,
        False,
        True,
        True,
        False
    ]
})

herramientas

### Interpretación

Las herramientas cubren problemas distintos.

Ninguna sustituye por sí sola a las demás.

## 22. Checklist de calidad

- [ ] ¿Cada función crítica tiene tests?
- [ ] ¿Hay casos límite?
- [ ] ¿Se testean excepciones?
- [ ] ¿Se usan fixtures cuando reducen duplicación?
- [ ] ¿Se parametrizan casos similares?
- [ ] ¿Hay tests de integración?
- [ ] ¿Existe algún test de regresión?
- [ ] ¿Se comparan DataFrames correctamente?
- [ ] ¿Se detectan efectos secundarios?
- [ ] ¿Se ejecuta Ruff?
- [ ] ¿El formato está automatizado?
- [ ] ¿Los type hints son verificables?
- [ ] ¿La configuración está en `pyproject.toml`?
- [ ] ¿El paquete es instalable?
- [ ] ¿Existe un comando o script de calidad?

## 23. Ejercicios finales

### Ejercicio 1 — Bug
Crea una función con un bug lógico y escribe un test que lo detecte.

### Ejercicio 2 — Excepción
Testea un `ValueError` con `pytest.raises`.

### Ejercicio 3 — Parametrización
Convierte cinco tests similares en uno parametrizado.

### Ejercicio 4 — Fixture
Crea una fixture con un DataFrame reutilizable.

### Ejercicio 5 — DataFrame
Utiliza `assert_frame_equal`.

### Ejercicio 6 — Regresión
Crea un test para un bug corregido y vuelve a introducirlo.

### Ejercicio 7 — Integración
Testea un pipeline de tres funciones.

### Ejercicio 8 — Mutación
Detecta una función que modifica el DataFrame original.

### Ejercicio 9 — Ruff
Introduce imports no usados y corrígelos.

### Ejercicio 10 — Black
Introduce formato inconsistente y automatiza su corrección.

### Ejercicio 11 — mypy
Introduce un retorno incompatible con el type hint.

### Ejercicio 12 — pyproject
Configura pytest, Ruff, Black y mypy.

### Ejercicio 13 — Packaging
Prepara un paquete `src/` instalable.

### Ejercicio 14 — Script de calidad
Crea un script que ejecute todas las comprobaciones.

### Ejercicio 15 — Reto final
Construye un proyecto completo con:

1. paquete `src/`;
2. al menos 8 funciones;
3. tests unitarios;
4. tests de integración;
5. fixtures;
6. parametrización;
7. tests de excepciones;
8. test de regresión;
9. test de DataFrame;
10. Ruff;
11. Black;
12. mypy;
13. `pyproject.toml`;
14. instalación editable;
15. script de calidad;
16. al menos un fallo provocado y corregido.

## 24. Preguntas de reflexión

1. ¿Qué demuestra realmente un test?
2. ¿Qué diferencia hay entre test unitario e integración?
3. ¿Por qué un test que falla puede ser útil?
4. ¿Qué aporta parametrización?
5. ¿Cuándo conviene una fixture?
6. ¿Por qué testear excepciones?
7. ¿Qué es un test de regresión?
8. ¿Por qué cobertura no equivale a calidad?
9. ¿Qué diferencia hay entre Ruff y Black?
10. ¿Qué detecta mypy?
11. ¿Los type hints impiden ejecutar código incorrecto?
12. ¿Qué aporta `pyproject.toml`?
13. ¿Qué significa que un paquete sea instalable?
14. ¿Por qué automatizar calidad?
15. ¿Qué debería ocurrir si una comprobación de calidad falla?